In [1]:
import pandas as pd

# El archivo debe tener los encabezados en la primera fila:
# Nombre | Ciudad | Estado
locations = pd.read_excel(
    'ArchivosExcel/Lugares.xlsx'
)

# Verificar las columnas disponibles
print("Columnas encontradas:", locations.columns.tolist())

if 'Nombre' not in locations.columns:
    raise ValueError("El archivo debe contener una columna llamada 'Nombre'.")

locations.head()


Columnas encontradas: ['País LATAM', 'Nombre', 'Ciudad', 'Estado / Provincia / Departamento']


,País LATAM,Nombre,Ciudad,Estado / Provincia / Departamento
0,MX,IMSS Hospital Centro Medico Siglo XXI,NaN,NaN
1,MX,Ángeles Metropolitano,Ciudad de México,NaN
2,MX,Hospital de alta especialidad del Bajío,NaN,Guanajuato


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import quote_plus
import re
import time

# Iniciar una sesión nueva de Chrome.
options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
options.add_argument('--window-size=1920,1080')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 12)


In [3]:
def clean_value(value):
    # Convierte valores vacíos/NaN a cadena vacía
    if pd.isna(value):
        return ''
    return str(value).strip()


def build_search_query(row):
    # Construye la búsqueda usando únicamente los campos disponibles.
    parts = [
        clean_value(row.get('Nombre', '')),
        clean_value(row.get('Ciudad', '')),
        clean_value(row.get('Estado', ''))
    ]
    return ' '.join(part for part in parts if part)


def get_google_name(driver):
    # Intenta obtener el nombre que muestra la ficha de Google Maps.
    selectors = [
        'h1.DUwDvf',
        'h1[class*="fontHeadlineLarge"]',
        'h1'
    ]

    for selector in selectors:
        try:
            element = WebDriverWait(driver, 3).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, selector))
            )
            value = element.text.strip()
            if value:
                return value
        except Exception:
            pass

    return 'Name not found'


def get_address_from_maps(driver):
    # Extrae la dirección visible en la ficha del lugar.
    selectors = [
        '[data-item-id="address"]',
        'button[data-item-id="address"]',
        'button[aria-label^="Address:"]',
        'button[aria-label^="Dirección:"]'
    ]

    for selector in selectors:
        try:
            element = WebDriverWait(driver, 4).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, selector))
            )

            value = element.get_attribute('aria-label') or element.text

            if value:
                value = re.sub(
                    r'^(Address|Dirección):\s*',
                    '',
                    value,
                    flags=re.IGNORECASE
                ).strip()

                if value:
                    return value

        except Exception:
            pass

    return 'Address not found'


def extract_postal_code(address):
    # Extrae un código postal de 5 dígitos de la dirección.
    # Está pensado principalmente para direcciones de México.
    if not address or address == 'Address not found':
        return 'Postal code not found'

    match = re.search(r'(?<!\d)(\d{5})(?:-\d{4})?(?!\d)', address)

    if match:
        return match.group(1)

    return 'Postal code not found'


def valid_coordinates(latitude, longitude):
    try:
        latitude = float(latitude)
        longitude = float(longitude)
        return -90 <= latitude <= 90 and -180 <= longitude <= 180
    except (TypeError, ValueError):
        return False


def extract_exact_coordinates_from_url(url):
    # Coordenadas exactas del lugar cuando Maps las incluye como !3dLAT!4dLON.
    if not url:
        return None, None

    match = re.search(
        r'!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)',
        url
    )

    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            return lat, lon

    return None, None


def extract_viewport_center(url):
    # Las coordenadas después de @ son el centro del mapa y NO siempre el lugar.
    # Se usan para elegir entre varios candidatos o como último respaldo.
    if not url:
        return None, None

    match = re.search(
        r'@(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)',
        url
    )

    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            return lat, lon

    return None, None


def _normalize_page_source(source):
    # Google incrusta partes de sus datos con comillas escapadas.
    # Normalizarlas permite detectar "latitude" y "longitude" de varias formas.
    if not source:
        return ''

    normalized = source

    # Puede haber uno o dos niveles de escape según cómo Maps serialice el dato.
    for _ in range(2):
        normalized = normalized.replace('\\\"', '"')
        normalized = normalized.replace('\\"', '"')

    replacements = {
        r'\u0022': '"',
        r'\u003d': '=',
        r'\u0026': '&',
        '&quot;': '"',
        '&#34;': '"'
    }

    for old, new in replacements.items():
        normalized = normalized.replace(old, new)

    return normalized


def extract_structured_coordinates_from_page(driver):
    # Estrategia principal de V3:
    # Google Maps suele incluir coordenadas en los datos internos de la ficha,
    # incluso cuando driver.current_url no contiene !3d/!4d.
    try:
        source = _normalize_page_source(driver.page_source)
    except Exception:
        return []

    candidates = []

    # Casos del tipo: "latitude":19.43,"longitude":-99.13
    patterns = [
        r'"latitude"\s*:\s*(-?\d+(?:\.\d+)?)\s*,\s*"longitude"\s*:\s*(-?\d+(?:\.\d+)?)',
        r'"latitude"\s*:\s*(-?\d+(?:\.\d+)?).{0,180}?"longitude"\s*:\s*(-?\d+(?:\.\d+)?)',
        # Algunas estructuras pueden guardar primero longitude.
        r'"longitude"\s*:\s*(-?\d+(?:\.\d+)?).{0,180}?"latitude"\s*:\s*(-?\d+(?:\.\d+)?)'
    ]

    for i, pattern in enumerate(patterns):
        for match in re.finditer(pattern, source, flags=re.DOTALL):
            if i < 2:
                lat, lon = float(match.group(1)), float(match.group(2))
            else:
                lon, lat = float(match.group(1)), float(match.group(2))

            if valid_coordinates(lat, lon):
                candidates.append((lat, lon))

    # También buscar !3d/!4d dentro del HTML completo, no solo en la URL.
    for match in re.finditer(
        r'!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)',
        source
    ):
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            candidates.append((lat, lon))

    # Eliminar duplicados conservando el orden.
    unique = []
    seen = set()
    for lat, lon in candidates:
        key = (round(lat, 7), round(lon, 7))
        if key not in seen:
            seen.add(key)
            unique.append((lat, lon))

    return unique


def choose_best_candidate(candidates, driver):
    if not candidates:
        return None, None

    if len(candidates) == 1:
        return candidates[0]

    # Si hay varias coordenadas embebidas (por ejemplo lugares cercanos),
    # elegir la más cercana al centro visible del mapa.
    center_lat, center_lon = extract_viewport_center(driver.current_url)

    if center_lat is None or center_lon is None:
        return candidates[0]

    def squared_distance(candidate):
        lat, lon = candidate
        return (lat - center_lat) ** 2 + (lon - center_lon) ** 2

    return min(candidates, key=squared_distance)


def get_coordinates_from_maps(driver, result_url=None):
    # PRIORIDAD V3:
    # 1) !3d/!4d del href real del resultado
    # 2) !3d/!4d de la URL final
    # 3) latitude/longitude o !3d/!4d encontrados dentro del HTML de la ficha
    # 4) @LAT,LON solo como último respaldo (centro del mapa)

    for label, candidate_url in [
        ('RESULT_URL_EXACT', result_url),
        ('CURRENT_URL_EXACT', driver.current_url)
    ]:
        lat, lon = extract_exact_coordinates_from_url(candidate_url)
        if lat is not None:
            return lat, lon, label

    # Esperar un poco a que aparezcan los datos internos de la ficha.
    end_time = time.time() + 6
    page_candidates = []

    while time.time() < end_time:
        page_candidates = extract_structured_coordinates_from_page(driver)
        if page_candidates:
            break
        time.sleep(0.5)

    if page_candidates:
        lat, lon = choose_best_candidate(page_candidates, driver)
        return lat, lon, 'PAGE_SOURCE'

    # Último respaldo: centro del mapa. Se marca claramente porque puede no ser exacto.
    for candidate_url in [driver.current_url, result_url]:
        lat, lon = extract_viewport_center(candidate_url)
        if lat is not None:
            return lat, lon, 'VIEWPORT_FALLBACK_REVIEW'

    return None, None, 'NOT_FOUND'


def click_first_result_if_needed(driver):
    # Si Google Maps devuelve una lista de resultados, guarda primero el href
    # del primer resultado y luego abre su ficha.
    try:
        results = driver.find_elements(
            By.CSS_SELECTOR,
            'a[href*="/maps/place/"]'
        )

        visible_results = [
            element for element in results
            if element.is_displayed()
        ]

        if visible_results:
            first_result = visible_results[0]
            result_url = first_result.get_attribute('href')
            url_before_click = driver.current_url

            driver.execute_script(
                "arguments[0].click();",
                first_result
            )

            try:
                WebDriverWait(driver, 7).until(
                    lambda d: (
                        d.current_url != url_before_click
                        or len(d.find_elements(
                            By.CSS_SELECTOR,
                            '[data-item-id="address"], h1.DUwDvf'
                        )) > 0
                    )
                )
            except Exception:
                pass

            time.sleep(1.2)
            return result_url
    except Exception:
        pass

    return None


def search_google_maps(row):
    # Busca un registro de Excel en Google Maps y devuelve
    # nombre, dirección, CP, coordenadas, fuente de coordenadas, query y estado.
    query = build_search_query(row)

    if not query:
        return {
            'Google_Name': '',
            'Google_Address': '',
            'Postal_Code': '',
            'Latitude': None,
            'Longitude': None,
            'Coordinate_Source': 'NOT_FOUND',
            'Search_Query': '',
            'Search_Status': 'EMPTY INPUT'
        }

    print(f'Buscando: {query}')

    try:
        url = (
            'https://www.google.com/maps/search/'
            f'?api=1&query={quote_plus(query)}'
        )

        driver.get(url)

        WebDriverWait(driver, 12).until(
            EC.presence_of_element_located((By.TAG_NAME, 'body'))
        )

        time.sleep(1.5)

        body_text = driver.find_element(By.TAG_NAME, 'body').text.lower()

        if (
            'unusual traffic' in body_text
            or 'tráfico inusual' in body_text
            or 'captcha' in body_text
        ):
            return {
                'Google_Name': '',
                'Google_Address': '',
                'Postal_Code': '',
                'Latitude': None,
                'Longitude': None,
                'Coordinate_Source': 'NOT_FOUND',
                'Search_Query': query,
                'Search_Status': 'MANUAL REVIEW - CAPTCHA'
            }

        result_url = None
        address = get_address_from_maps(driver)

        if address == 'Address not found':
            result_url = click_first_result_if_needed(driver)
            if result_url:
                address = get_address_from_maps(driver)

        google_name = get_google_name(driver)
        postal_code = extract_postal_code(address)

        latitude, longitude, coordinate_source = get_coordinates_from_maps(
            driver,
            result_url=result_url
        )

        if address == 'Address not found':
            status = 'REVIEW'
        elif latitude is None or longitude is None:
            status = 'ADDRESS FOUND - COORDINATES REVIEW'
        elif coordinate_source == 'VIEWPORT_FALLBACK_REVIEW':
            status = 'COORDINATES FROM MAP CENTER - REVIEW'
        elif postal_code == 'Postal code not found':
            status = 'ADDRESS FOUND - CP REVIEW'
        else:
            status = 'OK'

        print(f'  Nombre: {google_name}')
        print(f'  Dirección: {address}')
        print(f'  CP: {postal_code}')
        print(f'  Latitud: {latitude}')
        print(f'  Longitud: {longitude}')
        print(f'  Fuente coordenadas: {coordinate_source}')
        print(f'  Estado: {status}')

        time.sleep(1)

        return {
            'Google_Name': google_name,
            'Google_Address': address,
            'Postal_Code': postal_code,
            'Latitude': latitude,
            'Longitude': longitude,
            'Coordinate_Source': coordinate_source,
            'Search_Query': query,
            'Search_Status': status
        }

    except Exception as e:
        print(f'Error con {query}: {e}')

        return {
            'Google_Name': '',
            'Google_Address': 'Error',
            'Postal_Code': '',
            'Latitude': None,
            'Longitude': None,
            'Coordinate_Source': 'NOT_FOUND',
            'Search_Query': query,
            'Search_Status': 'ERROR'
        }


### Mejora en la extracción de coordenadas
Esta versión guarda el enlace (`href`) del resultado de Google Maps antes de abrirlo y busca las coordenadas primero ahí. Si no aparecen, utiliza la URL final del navegador como respaldo. Esto reduce los casos en los que la dirección se encuentra pero la latitud/longitud quedan vacías.


### Versión 3: extracción robusta de coordenadas

Esta versión ya no depende únicamente de la URL. Busca las coordenadas exactas en el `href`, en la URL final y, principalmente, en los datos internos (`page_source`) que Google Maps carga para la ficha. La columna `Coordinate_Source` indica qué método funcionó. Si aparece `VIEWPORT_FALLBACK_REVIEW`, las coordenadas vienen del centro visible del mapa y conviene revisarlas.


In [4]:
# Procesar todas las filas del Excel.
# Esta parte escala automáticamente al número de registros del archivo.
results = locations.apply(
    search_google_maps,
    axis=1
)

# Agregar los resultados al DataFrame original.
locations['Google_Name'] = results.apply(lambda x: x['Google_Name'])
locations['Google_Address'] = results.apply(lambda x: x['Google_Address'])
locations['Postal_Code'] = results.apply(lambda x: x['Postal_Code'])
locations['Latitude'] = results.apply(lambda x: x['Latitude'])
locations['Longitude'] = results.apply(lambda x: x['Longitude'])
locations['Coordinate_Source'] = results.apply(lambda x: x['Coordinate_Source'])
locations['Search_Query'] = results.apply(lambda x: x['Search_Query'])
locations['Search_Status'] = results.apply(lambda x: x['Search_Status'])


Buscando: IMSS Hospital Centro Medico Siglo XXI
  Nombre: Centro Médico Nacional Siglo XXI
  Dirección: Av. Cuauhtémoc 330, Doctores, Cuauhtémoc, 06720 Ciudad de México, CDMX
  CP: 06720
  Latitud: 19.4083431
  Longitud: -99.1539944
  Fuente coordenadas: RESULT_URL_EXACT
  Estado: OK
Buscando: Ángeles Metropolitano Ciudad de México
  Nombre: Hospital Angeles Metropolitano
  Dirección: Tlacotalpan 59, Centro Urbano Pdte. Juárez, Roma Sur, Cuauhtémoc, 06760 Cuauhtémoc, CDMX
  CP: 06760
  Latitud: 19.4067971
  Longitud: -99.1668709
  Fuente coordenadas: PAGE_SOURCE
  Estado: OK
Buscando: Hospital de alta especialidad del Bajío
  Nombre: Hospital Regional de Alta Especialidad del Bajío
  Dirección: Blvd. Milenio 130, San Carlos la Roncha, 37544 León de los Aldama, Gto.
  CP: 37544
  Latitud: 21.0613592
  Longitud: -101.5766002
  Fuente coordenadas: PAGE_SOURCE
  Estado: OK


In [5]:
# Revisar resultados antes de exportar
locations[
    [
        'Nombre',
        'Google_Name',
        'Google_Address',
        'Postal_Code',
        'Latitude',
        'Longitude',
        'Coordinate_Source',
        'Search_Status'
    ]
].head(20)


,Nombre,Google_Name,Google_Address,Postal_Code,Latitude,Longitude,Coordinate_Source,Search_Status
0,IMSS Hospital Centro Medico Siglo XXI,Centro Médico Nacional Siglo XXI,"Av. Cuauhtémoc 330, Doctores, Cuauhtémoc, 0672...",06720,19.408343,-99.153994,RESULT_URL_EXACT,OK
1,Ángeles Metropolitano,Hospital Angeles Metropolitano,"Tlacotalpan 59, Centro Urbano Pdte. Juárez, Ro...",06760,19.406797,-99.166871,PAGE_SOURCE,OK
2,Hospital de alta especialidad del Bajío,Hospital Regional de Alta Especialidad del Bajío,"Blvd. Milenio 130, San Carlos la Roncha, 37544...",37544,21.061359,-101.576600,PAGE_SOURCE,OK


In [6]:
# Cerrar Chrome después de procesar todas las ubicaciones
driver.quit()

# Guardar el archivo enriquecido
locations.to_excel(
    'ArchivosExcel/DireccionesRevisadas.xlsx',
    index=False
)

print('Archivo guardado en: ArchivosExcel/DireccionesRevisadas.xlsx')


PermissionError: [Errno 13] Permission denied: 'ArchivosExcel/DireccionesRevisadas.xlsx'